# Интерактивный дашборд результатов: Высоцкое городское поселение

Этот ноутбук **отдельный** от основного исследования. Он берёт уже посчитанный файл
`notebooks/data/vysokoye/results.geojson` и собирает автономную HTML-страницу с картой
показателей.

Что умеет дашборд:

- минималистичный интерфейс в стиле Apple;
- одна активная шкала — только для выбранного показателя;
- суммарные, медианные и другие метрики по каждому показателю;
- переключение подложек без API-ключей и режим «без подложки» (суша и вода из полигонов);
- сервисы OSM точками, с автоматическим включением нужного слоя для показателей обеспеченности;
- переключатель «Только кварталы / Без кварталов» — карта с кварталами или чистая подложка;
- режим «Доступность до квартала» с переключателем «Пешком / Авто»: выбор квартала
  и время в пути до него из всех остальных кварталов; правая кнопка мыши возвращает
  чистую карту.

Результат сохраняется в `notebooks/data/vysokoye/dashboard.html`.

**Важно:** сначала должен быть выполнен основной ноутбук `vysokoye_research.ipynb`,
чтобы появились `results.geojson`, `accessibility_matrix_pedestrian.csv` и
`accessibility_matrix_drive.csv`.


### 0. Настройка

Запустите эту ячейку первой.

In [1]:
# ============================================================
# 0. НАСТРОЙКА
# ============================================================
import sys
import warnings
from pathlib import Path

import geopandas as gpd
import pandas as pd

warnings.filterwarnings("ignore")


def _project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return cwd.parent if cwd.name == "notebooks" else cwd


PROJECT_ROOT = _project_root()
RESEARCH_DIR = PROJECT_ROOT / "notebooks" / "data" / "vysokoye"
BETTER_DATA_DIR = PROJECT_ROOT / "notebooks" / "better_data"
RESULTS_PATH = RESEARCH_DIR / "results.geojson"
HTML_PATH = RESEARCH_DIR / "dashboard.html"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from my_package.dashboard import build_dashboard  # noqa: E402

print("Файл результатов:", RESULTS_PATH)
print("Файл дашборда:", HTML_PATH)


Файл результатов: C:\TEMPLATE_AI_NIR\python-package-boilerplate\notebooks\data\vysokoye\results.geojson
Файл дашборда: C:\TEMPLATE_AI_NIR\python-package-boilerplate\notebooks\data\vysokoye\dashboard.html


### 1. Загрузка результатов

Читаем итоговый слой, посчитанный в основном ноутбуке.

In [2]:
# ============================================================
# 1. ЗАГРУЗКА РЕЗУЛЬТАТОВ
# ============================================================
if not RESULTS_PATH.exists():
    raise FileNotFoundError(
        "Нет файла results.geojson. Сначала выполните основной ноутбук vysokoye_research.ipynb."
    )

results = gpd.read_file(RESULTS_PATH).reset_index(drop=True).to_crs(4326)
print("Кварталов:", len(results))
print("Показателей:", len([c for c in results.columns if c != "geometry"]))
results.head()


Кварталов: 473
Показателей: 219


,residential,business,recreation,industrial,transport,special,agriculture,share,block_id,source_index,...,service_capacity_grocery,access_time_parks,service_demand_parks,service_served_parks,service_capacity_parks,has_building,has_population,graph_snap_drive_m,graph_snap_walk_m,geometry
0,0.0,0.0,1.000000,0.0,0.0,0.0,0.0,1.000000,B001,162,...,0.0,151.874437,0.0,0.0,0.0,False,False,5914.412983,3821.100798,"POLYGON ((28.59346 60.5245, 28.59353 60.52443,..."
1,0.0,0.0,0.999993,0.0,0.0,0.0,0.0,0.999993,B002,163,...,0.0,144.629379,0.0,0.0,0.0,False,False,5310.658150,3258.683784,"POLYGON ((28.58632 60.52793, 28.58627 60.52798..."
2,0.0,0.0,1.000000,0.0,0.0,0.0,0.0,1.000000,B003,164,...,0.0,143.658059,0.0,0.0,0.0,False,False,5229.714675,3338.096714,"POLYGON ((28.56956 60.53372, 28.5693 60.53372,..."
3,0.0,0.0,1.000000,0.0,0.0,0.0,0.0,1.000000,B004,165,...,0.0,142.791645,0.0,0.0,0.0,False,False,5157.513573,3246.283522,"POLYGON ((28.57153 60.53385, 28.5715 60.53379,..."
4,0.0,0.0,0.999883,0.0,0.0,0.0,0.0,0.999883,B005,166,...,0.0,129.456469,0.0,0.0,0.0,False,False,4046.248824,2256.558371,"POLYGON ((28.58021 60.53433, 28.58031 60.53427..."


### 2. Краткая сводка показателей

Эта таблица остаётся в ноутбуке для быстрой проверки данных.
В самом дашборде метрики пересчитываются и показываются для каждого выбранного показателя.

In [3]:
# ============================================================
# 2. Сводка
# ============================================================
base_numeric_cols = [
    "population", "fsi", "gsi", "mxi", "l", "osr", "area_accessibility",
    "green_share", "forest_share", "park_share", "green_m2_per_capita",
    "green_access_m", "green_access_500",
]
provision_cols = [c for c in results.columns if c.startswith("provision_strong_")]
numeric_cols = [c for c in base_numeric_cols + provision_cols if c in results.columns]

summary = pd.DataFrame({
    "Медиана": results[numeric_cols].median(),
    "Среднее": results[numeric_cols].mean(),
    "Максимум": results[numeric_cols].max(),
    "Кварталов": results[numeric_cols].count(),
})
summary


,Медиана,Среднее,Максимум,Кварталов
population,0.000000,2.399577,201.000000,473
fsi,0.000000,0.021369,0.624900,473
gsi,0.000000,0.018826,0.341480,473
mxi,0.000000,0.222060,0.800000,145
l,1.000000,1.069884,6.000000,145
osr,21.010658,546.254858,48006.991762,145
area_accessibility,62.723437,65.450668,110.121063,473
green_share,0.463255,0.498449,1.000000,473
forest_share,0.000000,0.346530,1.000000,473
park_share,0.000000,0.005173,0.910669,473


### 3. Сборка дашборда

Генератор находится в `my_package/dashboard.py`. Он встраивает данные в один HTML-файл:
кварталы, воду и сервисные точки. Поэтому страница открывается локально, а из сети
подгружаются только Leaflet и растровые подложки без API-ключей.

Доступные подложки: **Светлая**, **Улицы**, **Тёмная**, **Спутник** и **Без подложки**.
В режиме без подложки показываются суша и вода из полигонов.

In [4]:
# ============================================================
# 3. Сборка и сохранение
# ============================================================
output = build_dashboard(
    results_path=RESULTS_PATH,
    output_path=HTML_PATH,
    boundaries_path=BETTER_DATA_DIR / "boundaries.geojson",
    lakes_path=BETTER_DATA_DIR / "lake.geojson",
    services_dir=BETTER_DATA_DIR / "services",
    title="Высоцкое городское поселение",
    subtitle="Интерактивный профиль городской среды",
)

print("Дашборд сохранён:", output)
print(f"Размер файла: {output.stat().st_size / 1024 / 1024:.2f} МБ")
print("Откройте файл в браузере, чтобы посмотреть карту.")


2026-10-28 18:57:10.953 | WARNING  | blocksnet.utils.validation.gdf_schema:_warn_crs:40 - Current CRS 4326 is not projected. It might cause problems when carrying out spatial operations. Recommended: EPSG:32635


2026-10-28 18:57:11.055 | WARNING  | blocksnet.utils.validation.gdf_schema:_warn_crs:40 - Current CRS 4326 is not projected. It might cause problems when carrying out spatial operations. Recommended: EPSG:32635


2026-10-28 18:57:11.155 | WARNING  | blocksnet.utils.validation.gdf_schema:_warn_crs:40 - Current CRS 4326 is not projected. It might cause problems when carrying out spatial operations. Recommended: EPSG:32635


2026-10-28 18:57:11.234 | WARNING  | blocksnet.utils.validation.gdf_schema:_warn_crs:40 - Current CRS 4326 is not projected. It might cause problems when carrying out spatial operations. Recommended: EPSG:32635


2026-10-28 18:57:11.317 | WARNING  | blocksnet.utils.validation.gdf_schema:_warn_crs:40 - Current CRS 4326 is not projected. It might cause problems when carrying out spatial operations. Recommended: EPSG:32635


2026-10-28 18:57:11.433 | WARNING  | blocksnet.utils.validation.gdf_schema:_warn_crs:40 - Current CRS 4326 is not projected. It might cause problems when carrying out spatial operations. Recommended: EPSG:32635


... (6 outputs left out)

2026-10-28 18:57:12.081 | WARNING  | blocksnet.utils.validation.gdf_schema:_warn_crs:40 - Current CRS 4326 is not projected. It might cause problems when carrying out spatial operations. Recommended: EPSG:32635


2026-10-28 18:57:12.159 | WARNING  | blocksnet.utils.validation.gdf_schema:_warn_crs:40 - Current CRS 4326 is not projected. It might cause problems when carrying out spatial operations. Recommended: EPSG:32635


2026-10-28 18:57:12.244 | WARNING  | blocksnet.utils.validation.gdf_schema:_warn_crs:40 - Current CRS 4326 is not projected. It might cause problems when carrying out spatial operations. Recommended: EPSG:32635


Дашборд сохранён: C:\TEMPLATE_AI_NIR\python-package-boilerplate\notebooks\data\vysokoye\dashboard.html
Размер файла: 4.37 МБ
Откройте файл в браузере, чтобы посмотреть карту.


### 4. Как пользоваться

- **Показатель:** выберите значение в списке — карта, метрики и шкала обновятся одновременно.
- **Шкала:** показывается только для активного показателя. Для категориальных показателей
  вместо шкалы выводятся категории с количеством кварталов.
- **Подложка:** переключается справа сверху. «Без подложки» оставляет только сушу и воду
  из полигонов; все варианты работают без API-ключей.
- **Сервисы:** точки OSM можно включать вручную. Для показателей обеспеченности нужный
  слой включается автоматически.
- **Только кварталы / Без кварталов:** переключатель справа сверху показывает кварталы
  без тематической окраски или полностью скрывает их слой.
- **Доступность до квартала:** включите режим, выберите «Пешком» или «Авто», затем кликните
  квартал — карта покажет время в пути до него. Повторный клик меняет цель, ПКМ возвращает
  чистую карту.
- **Ссылка на показатель:** добавьте `#population` или `#provision_strong_kindergarten`
  к адресу страницы, чтобы открыть дашборд сразу на нужном показателе.

Если карта пустая — значит основной ноутбук ещё не посчитал `results.geojson`.
